In [ ]:
# Version v6.0.0
from datetime import datetime
#import hashlib
#import importlib.metadata
import json
from pathlib import Path
#from tempfile import NamedTemporaryFile
from dotenv import load_dotenv
import os
from pysr import PySRRegressor
from sklearn.utils.class_weight import compute_sample_weight
import matplotlib.pyplot as plt
#from matplotlib.patches import Rectangle
import numpy as np
import pandas as pd
import sympy as sp
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import roc_curve, auc

load_dotenv()
PROJECT_DIR = Path.cwd().resolve()

SEED = 42
VERSION = "6.0.0"

# MASS_LABELS = {"mchi1": r"$m_{\chi_1}$ [GeV]", "mchipm1": r"$m_{\chi^\pm_1}$ [GeV]"}


def run_id_ledger():
    counter_file = Path(".run_id_counter")

    while True:
        i = int(counter_file.read_text()) + 1 if counter_file.exists() else 1

        counter_file.write_text(str(i))

        yield i


# splitting made easy
def do_split(X, y, train_p=60, test_p=40, seed=SEED):
    if (train_p + test_p)!=100:
        raise ValueError 
    # Calculate splitting ratios
    test_size = (test_p) / 100
    
    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=test_size,
        stratify=y,
        random_state=seed,
    )

    X_total, y_total = X, y
    return X_train, y_train, X_test, y_test, X_total, y_total


def expr_indexer(HOF, RUN_ID, features):
    """Normalize model/CSV equations and assign RUN_ID_0{complexity} IDs."""
    # features = tuple(features)
    # HOF_01 = HOF.rename(columns={"complexity": "Complexity", "loss": "Loss", "equation": "Equation",
    #                             "sympy_format": "SymPy_Equation", "lambda_format": "Lambda_Equation"}).copy()

    # HOF_01["Complexity"] = complexity.astype(int)
    # HOF_01["Loss"] = pd.to_numeric(HOF_01["Loss"], errors="raise")
    # HOF_01.insert(0, "Expression_ID", [f"{RUN_ID}_0{c}" for c in HOF_01["Complexity"]])
    # symbols = dict(zip(features, map(sp.Symbol, features)))
    # expressions, functions = [], []
    # for _, row in HOF_01.iterrows():
    #     expr = sp.sympify(row.get("SymPy_Equation", row["Equation"]),
    #                       locals={**symbols, "square": lambda value: value**2,
    #                               "cube": lambda value: value**3}, evaluate=False)
    #     if {str(symbol) for symbol in expr.free_symbols} - set(features):
    #         raise ValueError("Expression contains symbols outside the supplied features.")
    #     fn = row.get("Lambda_Equation")
    #     if not callable(fn):
    #         numeric = sp.lambdify(list(symbols.values()), expr, modules="numpy")
    #         fn = lambda X, numeric=numeric: numeric(*(X[name].to_numpy(dtype=float) for name in features))
    #     expressions.append(expr)
    #     functions.append(fn)
    # HOF_01["SymPy_Equation"], HOF_01["Lambda_Equation"] = expressions, functions
    pass #HOF_01


def load_saved_expression(source, expression_id, features=None):
    """Read one raw-run or curated expression without loading PySR/checkpoints.

    A manifest owns its run ID, feature order and relative hall-of-fame path.
    Its dataset hash describes the current reproduction, not historical membership.
    """
    pass




def write_run_metadata(run_id, *, run_dir, data_csv, datasets, model_parameters:dict,
                       selection_metric="test_auc",
                       split_ratio:str|tuple,
                       num_threads,
                       recorded_before_fit=True ,weight_policy=None):
    """Capture actual inputs; recipe fields omitted by the caller remain unknown."""
    run_dir_str = str(run_dir)
    data_csv_path = str(Path(data_csv).resolve())
    X_train, ___ = datasets["train"]
    document = {
        "version": VERSION, "run_id": run_id,
        "run_dir":run_dir_str,
        "data_path": data_csv_path,
        "created_at_utc": datetime.now().isoformat(), 
        "recorded_before_fit": recorded_before_fit,
        "features": list(X_train.columns), "target": "exclusion", "split_ratio": split_ratio,
        "training_label_mapping": {"0": -1, "1": 1}, "exclusion_labels": [0, 1],
        "weight_policy": weight_policy,
        "selection_metric": selection_metric,
        "num_threads": num_threads,
        "model_parameters": model_parameters
    }
    destination = Path(run_dir) / "run_metadata.json"
    with destination.open("w", encoding="utf-8") as f:
        json.dump(document, f, indent=4)
    return destination


/Users/vaheedgorgeen/libs/train-symexpr-bsm/.venv/lib/python3.13/site-packages/pysr/julia_import.py:53: UserWarning: PYTHON_JULIACALL_THREADS environment variable is set to something other than 'auto', so PySR was not able to set it. You may wish to set it to `'auto'` for full use of your CPU.
  warnings.warn(


In [2]:
print(os.getenv("PYTHON_JULIACALL_EXE"))
print(os.getenv("PYTHON_JULIACALL_PROJECT"))
if int(os.getenv('PYTHON_JULIACALL_THREADS'))>=1:
    num_threads=os.getenv('PYTHON_JULIACALL_THREADS')
    print(int(num_threads))

/Users/vaheedgorgeen/.local/opt/julia-1.13.1/julia-1.13.1/bin/julia
/Users/vaheedgorgeen/libs/train-symexpr-bsm/.venv/julia_env
4


In [11]:
# The unchanged run counter intentionally operates in the repository root.
if Path.cwd().resolve() != PROJECT_DIR:
    raise RuntimeError(f"Run main.py from {PROJECT_DIR}")

total_ME1 = "data/masses_exclusions.csv"
df = pd.read_csv(total_ME1)

features = ["mchi1","mchipm1"]
target = "exclusion"

df = df.dropna(subset=features + [target])

X = df[features].astype(float)
y = df[target].astype(int)
# Check if all y's are 0s or 1s

X_train, y_train, X_test, y_test, X_total, y_total = do_split(X,y, train_p=70, test_p=30, seed=SEED)
# Save a copy for later use
# cp_path.mkdir(parents=True, exist_ok=True)
# test_ME1 = X_test.copy()
# test_ME1["exclusion"] = np.asarray(y_test)
# test_ME1.to_csv(cp_path / "test_ME1.csv", index=False)

# val_ME1 = X_val.copy()
# val_ME1["exclusion"] = np.asarray(y_val)
# val_ME1.to_csv(cp_path / "val_ME1.csv", index=False)


y_train_signed = 2 * y_train.astype(int) - 1
sample_weights_signed = compute_sample_weight(class_weight="balanced", y=y_train_signed)
# Choose ONE:
loss: str

loss_index= ["LML", "ZOL", "L2HL", "L2ML", "MHL", "SL"]
loss_dict ={
    "loss_LML": "LogitMarginLoss()",
    "loss_ZOL": "ZeroOneLoss()",
    "loss_L2HL" : "L2HingeLoss()",
    "loss_L2ML": "L2MarginLoss()",
    "loss_MHL": "ModifiedHuberLoss()",
    "loss_SL": "SigmoidLoss()"
}
for _ in loss_index:
    if (_=="LML"):
        loss = loss_dict[f"loss_{_}"]


RUN_ID = f"RUN_{next(run_id_ledger())}_{loss.strip("()")}"

RUN_DIR = PROJECT_DIR / "output" / RUN_ID
RUN_DIR.mkdir(parents=True, exist_ok=False)

# NEW Regressor -> based on https://pysr.ai/v2.7.2/#detailed-example
model = PySRRegressor(
    elementwise_loss=loss,
    parallelism="multithreading",
    # procs=4,
    populations=8,
    # ^ Assuming we have 4 cores, this means 2 populations per core, so one is always running.
    population_size=100,
    # ^ Slightly larger populations, for greater diversity.
    ncycles_per_iteration=100,
    # ^ Generations between migrations.   ???
    niterations=100,
    early_stop_condition=(
        "stop_if(loss, complexity) = loss < 1e-6 && complexity < 500"
        # Stop early if we find a good and simple equation
    ),
    timeout_in_seconds=60 * 60 * 1,
    # ^ Alternatively, stop after (HP) 4 hours have passed.
    output_directory="output/",
    run_id=RUN_ID,
    maxsize=300,
    # ^ Allow greater complexity.
    #maxdepth=40,
    # ^ But, avoid deep nesting.
    binary_operators=["*", "+", "-", "/"],
    unary_operators=["square", "exp", "cos", "sinh"],
    # constraints={
    #     "/": (-1, 4),
    #     "square": 5,
    #     "cube": 2,
    #     "exp": 6,
    # },
    # ^ Limit the complexity within each argument.
    # "inv": (-1, 9) states that the numerator has no constraint,
    # but the denominator has a max complexity of 9.
    # "exp": 9 simply states that `exp` can only have
    # an expression of complexity 9 as input.
    # nested_constraints={
    #      "square": {"square": 1, "cube": 1, "exp": 0},
    #      "cube": {"square": 1, "cube": 1, "exp": 0},
    #      "exp": {"square": 1, "cube": 1, "exp": 0},
    # },
    # ^ Nesting constraints on operators. For example,
    # "square(exp(x))" is not allowed, since "square": {"exp": 0}.
    #complexity_of_operators={"/": 2, "exp": 3},
    # ^ Custom complexity of particular operators.
    #complexity_of_constants=2,
    # ^ Punish constants more than variables
    #select_k_features=2,
    # ^ Train on only the 4 most important features -> Here -> 2
    progress=True,
    # ^ Can set to false if printing to a file.
    #weight_randomize=0.1,
    # ^ Randomize the tree much more frequently
    # cluster_manager=None,
    # ^ Can be set to, e.g., "slurm", to run a slurm
    # cluster. Just launch one script from the head node.
    precision=64,
    # ^ Higher precision calculations.
    warm_start=True,
    # ^ Start from where left off.
    # turbo=True,
    # ^ Faster evaluation (experimental)
    # extra_sympy_mappings={"cos2": lambda x: sp.cos(x)**2},
    # extra_torch_mappings={sympy.cos: torch.cos},
    # ^ Not needed as cos already defined, but this
    # is how you define custom torch operators.
    # extra_jax_mappings={sympy.cos: "jnp.cos"},
    # ^ For JAX, one passes a string
    parsimony=1e-3,
    #adaptive_parsimony_scaling=40

)

datasets = {"train": (X_train, y_train), "test": (X_test, y_test), "total": (X_total, y_total)}



In [12]:
config_dest = write_run_metadata(
    run_id=RUN_ID, 
    run_dir=RUN_DIR, 
    data_csv=total_ME1, 
    datasets=datasets,
     num_threads=num_threads,
     model_parameters={
          "elementwise_loss": f"{loss}",
          "parallelism": "multithreading",
          "populations":8,
          "population_size":100,
          "ncycles_per_iteration":100,
          "niterations":"100",
          "early_stop_condition":"(stop_if(loss, complexity) = loss < 1e-6 && complexity < 500)",
          "timeout_in_seconds":60 * 60 * 1,
          "maxsize":300,
          #maxdepth=40,
          "binary_operators":["*", "+", "-", "/"],
          "unary_operators":["square", "exp", "cos", "sinh"],
          "progress":True,
          "precision":64,
          "warm_start":True,
          "parsimony":1e-3},
     selection_metric="test_auc",
     split_ratio=(70,30),
     recorded_before_fit=True, 
     weight_policy="sample_weights"
)

print(f"Experiment has been recorded in {str(config_dest)}")

Experiment has been recorded in /Users/vaheedgorgeen/libs/train-symexpr-bsm/output/RUN_3_LogitMarginLoss/run_metadata.json


In [13]:
model.fit(X_train,y_train_signed,weights=sample_weights_signed)

/Users/vaheedgorgeen/libs/train-symexpr-bsm/.venv/lib/python3.13/site-packages/pysr/sr.py:1971: UserWarning: Note: Using a large maxsize for the equation search will be exponentially slower and use significant memory.
  warnings.warn(
Compiling Julia backend...
[ Info: Started!



Expressions evaluated per second: 2.940e+03
Progress: 7 / 100 total iterations (7.875%)
════════════════════════════════════════════════════════════════════════════════════════════════════
───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Score      Equation
1           6.931e-01  0.000e+00  y = 1.2376e-07
3           6.367e-01  4.246e-02  y = -171.18 / mchipm1
5           6.193e-01  1.384e-02  y = -0.043158 / (-24.097 / mchi1)
6           6.041e-01  2.485e-02  y = sinh(-0.043158 / (-24.097 / mchi1))
7           5.817e-01  3.777e-02  y = (-0.043158 / (-24.097 / mchipm1)) + -0.31533
13          5.797e-01  5.845e-04  y = ((((mchipm1 / mchipm1) / (-24.097 / mchi1)) / -24.097)...
                                       / 0.41186) + -0.50527
15          5.562e-01  2.073e-02  y = sinh(cos((-4.2791 / ((-0.043318 / (mchipm1 / -0.99765)...
                                      ) + mchipm1)) / 0.010695)) - 0.54704
21       

[ Info: Final population:
[ Info: Results saved to:


───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Score      Equation
1           6.931e-01  0.000e+00  y = 5.9749e-09
3           6.213e-01  5.475e-02  y = mchi1 * 0.0014355
4           5.865e-01  5.765e-02  y = square(mchi1 * 0.0014355)
5           4.870e-01  1.859e-01  y = (mchipm1 / 308.14) + -1.6179
6           4.851e-01  3.780e-03  y = square(mchipm1 / 444.3) + -1.3964
7           4.653e-01  4.180e-02  y = (-0.22731 / (-30.982 / mchi1)) - 2.4159
8           4.585e-01  1.472e-02  y = square(-0.10891 / (-31.785 / mchi1)) - 1.4914
13          4.103e-01  2.219e-02  y = ((((mchi1 / -23.669) / (-37.083 / mchipm1)) / 575.7) /...
                                       0.16018) + -1.8948
14          4.063e-01  9.923e-03  y = sinh(((((mchipm1 / -23.67) / 0.22529) / (-31.954 / mch...
                                      i1)) / 612.78) + -1.5079)
34          3.855e-01  2.624e-03  y = sinh(((4.0718 - cos(mchi1 - mchipm1

PySRRegressor.equations_ = [
	    pick     score                                           equation  \
	0         0.000000                                5.97492670201204e-9   
	1         0.054746                      mchi1 * 0.0014355336931198136   
	2         0.057652              square(mchi1 * 0.0014355336931198136)   
	3         0.185877  (mchipm1 / 308.1416486921858) + -1.61786657261...   
	4         0.003780  square(mchipm1 / 444.3009495373577) + -1.39636...   
	5         0.041796  (-0.22730908322112933 / (-30.98154760098834 / ...   
	6         0.014718  square(-0.10890841116255451 / (-31.78451880180...   
	7         0.022189  ((((mchi1 / -23.669235398251494) / (-37.082687...   
	8         0.009923  sinh(((((mchipm1 / -23.67033472897696) / 0.225...   
	9         0.002624  sinh(((4.071762082803053 - cos(mchi1 - mchipm1...   
	10        0.002269  sinh(((4.071762082803053 - cos(mchi1 - mchipm1...   
	11        0.000570  sinh(((4.071762082803053 - cos(mchi1 - mchipm1...   
	12      